# Despliegue en *Linux* por *SSH*.

No todo destino es un servicio gestionado. En muchos equipos la aplicación todavía corre en un servidor *Linux* al que se llega por *SSH* (*Secure Shell*). Este cuaderno practica un **primer despliegue remoto controlado**: autenticación por clave, verificación del anfitrión, copia de la aplicación, reconstrucción idempotente del servicio y una comprobación posterior.

## Objetivos.

* Preparar un destino *Linux* de laboratorio y una clave *SSH*, sin incrustar credenciales en el repositorio.
* Leer el *workflow* ```.github/workflows/deploy-linux-ssh.yml``` y ejecutarlo contra ese destino.
* Distinguir el valor por defecto de la práctica (máquina proporcionada por quien imparte el curso) de la variante con un contenedor local y un *runner* autoalojado.


## Valor por defecto de la práctica, y cómo cambiarlo.

**Valor por defecto.** El destino es una máquina *Linux* proporcionada por quien imparte el curso, alcanzable por *SSH* desde los *runners* hospedados de *GitHub*. El *workflow* vigente usa ```runs-on: ubuntu-latest```, copia el ```Dockerfile```, el ```.dockerignore``` y el directorio ```api/``` al servidor, construye la imagen allí y la levanta con *Docker Compose*.

**Variante.** Si se prefiere no exponer *SSH* a internet, el destino puede ser un contenedor con servidor *SSH* en la misma máquina que el *runner* autoalojado del cuaderno 10. En ese caso basta con:

* cambiar ```runs-on``` a ```self-hosted```;
* apuntar los secretos de conexión (```SSH_HOST```, ```SSH_PORT```, ```SSH_USER```, ```SSH_PRIVATE_KEY``` y ```SSH_KNOWN_HOSTS```) a ese contenedor.

El resto del *workflow* no cambia. Este material está escrito para el valor por defecto, que es el que implementa el archivo vigente.


## Preparación del destino y de las credenciales.

En el servidor de la práctica debe haber *Docker* y *Docker Compose*, y un directorio de trabajo que se indicará en la variable de repositorio ```SSH_DEPLOY_DIR```.

1. Crear un par de claves solo para esta práctica. El archivo de la clave privada **no** se versiona.

    ```bash
    mkdir -p "$HOME/cd441-destino/ssh"
    ssh-keygen -t ed25519 -f "$HOME/cd441-destino/ssh/deploy" -N ""
    ```

2. Instalar la clave pública en la cuenta de destino del servidor (el usuario que indique quien imparte el curso).

3. Obtener la entrada de ```known_hosts``` del servidor, para no desactivar ```StrictHostKeyChecking```:

    ```bash
    ssh-keyscan -p "$SSH_PORT" "$SSH_HOST"
    ```

    Esos valores de puerto y anfitrión se toman de la sesión de trabajo, no de un archivo del repositorio.

En el repositorio de *GitHub* se configuran los secretos ```SSH_HOST```, ```SSH_USER```, ```SSH_PORT```, ```SSH_PRIVATE_KEY``` y ```SSH_KNOWN_HOSTS```, y la variable ```SSH_DEPLOY_DIR```. El *workflow* exige ```SSH_KNOWN_HOSTS```: no acepta un destino cuya huella no se haya guardado de antemano.


## Controles mínimos.

Un despliegue por *SSH* no es «un comando remoto y ya». En este curso se exige:

* **Claves, no contraseñas**, y la clave privada solo como secreto de *GitHub*, nunca en el repositorio.
* **Verificación del anfitrión** con ```SSH_KNOWN_HOSTS``` y ```StrictHostKeyChecking=yes```.
* **Pasos idempotentes**: volver a ejecutar el *workflow* deja el servicio en el estado deseado. El manifiesto ```api/deploy/compose.yaml``` usa ```docker compose up -d --build```, que reconstruye y recrea el servicio de forma repetible.
* **Evidencia posterior**: una petición ```GET /health``` en el propio anfitrión.

El destino necesita *Docker* para construir y correr la imagen de la *API*. Una preparación más amplia del anfitrión (por ejemplo, un motor de base de datos *PostgreSQL* en el propio servidor) no es el centro de este cuaderno; si hiciera falta, quien imparte el curso indica los pasos mínimos, y los cursos posteriores profundizan en esa operación.

Hay también una unidad *systemd* de ejemplo en ```api/deploy/cd441-api.service```, por si el destino no usa *Compose*. El *workflow* vigente usa *Compose*.


## El *workflow* ```deploy-linux-ssh.yml```.

El archivo vigente es ```.github/workflows/deploy-linux-ssh.yml```. Se dispara solo de forma manual (*workflow_dispatch*). Los pasos esenciales, alineados con ese archivo, son:

```yaml
name: Despliegue en Linux por SSH

on:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  desplegar:
    runs-on: ubuntu-latest
    env:
      SSH_HOST: ${{ secrets.SSH_HOST }}
      SSH_USER: ${{ secrets.SSH_USER }}
      SSH_PORT: ${{ secrets.SSH_PORT }}
      SSH_PRIVATE_KEY: ${{ secrets.SSH_PRIVATE_KEY }}
      SSH_KNOWN_HOSTS: ${{ secrets.SSH_KNOWN_HOSTS }}
      SSH_DEPLOY_DIR: ${{ vars.SSH_DEPLOY_DIR }}
      APP_VERSION: ${{ github.sha }}
    steps:
    - name: Checkout
      uses: actions/checkout@v7

    - name: Preparar clave y known_hosts
      run: |
        mkdir -p "${HOME}/.ssh"
        chmod 700 "${HOME}/.ssh"
        printf '%s\n' "${SSH_PRIVATE_KEY}" > "${HOME}/.ssh/id_ed25519"
        chmod 600 "${HOME}/.ssh/id_ed25519"
        printf '%s\n' "${SSH_KNOWN_HOSTS}" > "${HOME}/.ssh/known_hosts"
        chmod 644 "${HOME}/.ssh/known_hosts"

    - name: Copiar la aplicación al servidor
      run: |
        PUERTO="${SSH_PORT:-22}"
        DESTINO="${SSH_USER}@${SSH_HOST}"
        OPCIONES=(-i "${HOME}/.ssh/id_ed25519" -o IdentitiesOnly=yes -o StrictHostKeyChecking=yes)
        ssh "${OPCIONES[@]}" -p "${PUERTO}" "${DESTINO}" "mkdir -p '${SSH_DEPLOY_DIR}'"
        scp "${OPCIONES[@]}" -P "${PUERTO}" -r Dockerfile .dockerignore api "${DESTINO}:${SSH_DEPLOY_DIR}/"

    - name: Construir, arrancar y comprobar
      run: |
        PUERTO="${SSH_PORT:-22}"
        DESTINO="${SSH_USER}@${SSH_HOST}"
        OPCIONES=(-i "${HOME}/.ssh/id_ed25519" -o IdentitiesOnly=yes -o StrictHostKeyChecking=yes)
        ssh "${OPCIONES[@]}" -p "${PUERTO}" "${DESTINO}" \
          "cd '${SSH_DEPLOY_DIR}' && export APP_VERSION='${APP_VERSION}' && docker compose -f api/deploy/compose.yaml up -d --build && curl -fsS http://127.0.0.1:8080/health"
```

El *workflow* comprueba, al inicio, que los secretos de conexión y la variable ```SSH_DEPLOY_DIR``` estén definidos. La ruta de salud es ```GET /health```. La versión desplegada viaja en ```APP_VERSION```, igual al *SHA* del *commit*.


## Actividad.

1. Preparar el destino por defecto (máquina indicada por quien imparte el curso) o, si se usa la variante, el contenedor local y el *runner* autoalojado.
2. Cargar los secretos de conexión y la variable ```SSH_DEPLOY_DIR``` en el repositorio propio.
3. Disparar el *workflow* ```Despliegue en Linux por SSH``` y comprobar que ```GET /health``` responde en el destino, con la versión del *commit*.
4. Volver a dispararlo sin cambiar el código y verificar que el resultado es el mismo (idempotencia).
5. Localizar en el *workflow* la línea ```runs-on``` que habría que cambiar para pasar de un *runner* hospedado a uno autoalojado.


## Cierre.

Con este cuaderno se cierra el tramo práctico de *cd441*: integración continua de una *API*, imagen en un registro, un primer despliegue en *Cloud Run* y un primer despliegue en *Linux* por *SSH*. Los cursos posteriores pueden dar esa práctica por cubierta y avanzar hacia infraestructura como código, federación de identidad y servicios de *GCP* que aquí solo se nombraron.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>